In [1]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["USE_TF"] = "0"   


In [2]:
import os
import pandas as pd

EVENT_FILES = {
    "Election2024": {
        "human": [
            "election2024_tweets.csv",
            "Elec2024Tel.csv"
        ],
        "ai": [
            "election2024_ai_generated.csv",
            "Election2024_synthetic_posts.csv"
        ]
    },

    "Election2020": {
        "human": ["election2020_tweets.csv"],
        "ai": ["election2020_toxic_synthetic.csv"]
    },
    "CapitolAttack":{
        "human":["Capitol_Attack202_tweets.csv",
                "CapitolTel.csv"],
        "ai":["Capitol_Attack_synthetic_posts.csv",
                "Capitol_Attack_synthetic_posts1.csv",
              "capitol_attack_toxic_synthetic.csv"
              ]
    },
    "Covid":{
        "human":["covid_tweets3.csv"],
        "ai":["covid_toxic_textonly.csv",
            "Covid_synthetic_posts.csv"]
    },
    "Summer2020":{
        "human":["Summer2020_tweets2.csv"],
        "ai":["Summer2020_synthetic_posts.csv"]
    },
    "Utah":{
        "human":["Utah_tweets2.csv"],
        "ai":["utah_shooting_toxic_textonly.csv",
            "Utah_Shooting_2025_Dataset.csv"]
    },
    "Roe":{
        "human":["Roe2022_tweets.csv"],
        "ai":["roe_wade_toxic_textonly.csv",
            "Roe2022_synthetic_posts.csv"]
    },
    "Midterm":{
        "human":["midterm_tweets224.csv"],
        "ai":["Midterm_synthetic_posts.csv"]
    },
    
}

OUT_DIR = "out"
os.makedirs(OUT_DIR, exist_ok=True)

TEXT_COL_CANDIDATES = ["text","Text","content","body","message","post","tweet"]

def find_text_col(df):
    for c in TEXT_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError("Could not find text column")

def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None

def load_one(path, label, event):
    df = pd.read_csv(path)

    text_col = find_text_col(df)
    df = df.rename(columns={text_col:"text"})

    df["text"] = df["text"].apply(clean_text)
    df = df.dropna(subset=["text"]).copy()

    df["label"] = label
    df["event"] = event

    df = df.drop_duplicates(subset=["text"])
    return df[["text","label","event"]]


all_human = []
all_ai = []

for event, paths in EVENT_FILES.items():

    print(f"\nProcessing event: {event}")

    # --- HUMAN FILES ---
    for p in paths["human"]:
        print("  human:", p)
        all_human.append(load_one(p, 0, event))

    # --- AI FILES ---
    for p in paths["ai"]:
        print("  ai:", p)
        all_ai.append(load_one(p, 1, event))


human_df = pd.concat(all_human, ignore_index=True)
ai_df = pd.concat(all_ai, ignore_index=True)

# Remove duplicates across whole dataset
human_df = human_df.drop_duplicates(subset=["text"])
ai_df = ai_df.drop_duplicates(subset=["text"])

combined = pd.concat([human_df, ai_df], ignore_index=True)
combined = combined.sample(frac=1.0, random_state=42).reset_index(drop=True)

human_df.to_csv("out/human_combined_with_event.csv", index=False)
ai_df.to_csv("out/ai_combined_with_event.csv", index=False)
combined.to_csv("out/combined_with_event.csv", index=False)

print("\nDONE")
print("Total:", len(combined))
print("\nEvents distribution:")
print(combined["event"].value_counts())


Processing event: Election2024
  human: election2024_tweets.csv
  human: Elec2024Tel.csv
  ai: election2024_ai_generated.csv
  ai: Election2024_synthetic_posts.csv

Processing event: Election2020
  human: election2020_tweets.csv
  ai: election2020_toxic_synthetic.csv

Processing event: CapitolAttack
  human: Capitol_Attack202_tweets.csv
  human: CapitolTel.csv
  ai: Capitol_Attack_synthetic_posts.csv
  ai: Capitol_Attack_synthetic_posts1.csv
  ai: capitol_attack_toxic_synthetic.csv

Processing event: Covid
  human: covid_tweets3.csv
  ai: covid_toxic_textonly.csv
  ai: Covid_synthetic_posts.csv

Processing event: Summer2020
  human: Summer2020_tweets2.csv
  ai: Summer2020_synthetic_posts.csv

Processing event: Utah
  human: Utah_tweets2.csv
  ai: utah_shooting_toxic_textonly.csv
  ai: Utah_Shooting_2025_Dataset.csv

Processing event: Roe
  human: Roe2022_tweets.csv
  ai: roe_wade_toxic_textonly.csv
  ai: Roe2022_synthetic_posts.csv

Processing event: Midterm
  human: midterm_tweets224

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
print("✅ transformers imported without TF")


/home/gg2713/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ transformers imported without TF


In [4]:
import sys, site, importlib.util

print("Python:", sys.executable)
print("User site:", site.getusersitepackages())
print("User site in sys.path?", site.getusersitepackages() in sys.path)

print("find_spec('sentencepiece') =", importlib.util.find_spec("sentencepiece"))

try:
    import sentencepiece
    print("✅ sentencepiece imported:", sentencepiece.__version__)
    print("sentencepiece file:", sentencepiece.__file__)
except Exception as e:
    print("❌ sentencepiece import failed:", e)

print("\nFirst 5 sys.path entries:\n", "\n".join(sys.path[:5]))



Python: /share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter/bin/python
User site: /home/gg2713/.local/lib/python3.12/site-packages
User site in sys.path? True
find_spec('sentencepiece') = ModuleSpec(name='sentencepiece', loader=<_frozen_importlib_external.SourceFileLoader object at 0x1554fe6f49e0>, origin='/home/gg2713/.local/lib/python3.12/site-packages/sentencepiece/__init__.py', submodule_search_locations=['/home/gg2713/.local/lib/python3.12/site-packages/sentencepiece'])
✅ sentencepiece imported: 0.2.1
sentencepiece file: /home/gg2713/.local/lib/python3.12/site-packages/sentencepiece/__init__.py

First 5 sys.path entries:
 /share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter/lib/python312.zip
/share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter/lib/python3.12
/share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter/lib/python3.12/lib-dynload

/home/gg2713/.local/lib/python3.12/site-packages


In [5]:
import site, sys
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.append(user_site)

import sentencepiece
print("✅ sentencepiece:", sentencepiece.__version__)



✅ sentencepiece: 0.2.1


In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base", use_fast=False)
print("✅ tokenizer loaded")


✅ tokenizer loaded


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = "out/combined_with_event.csv"
HOLDOUT_EVENT = "Summer2020"
SEED = 42

df = pd.read_csv(DATA_PATH)

assert {"text","label","event"}.issubset(df.columns)

# CLEAN HERE
df = df.dropna(subset=["text","label","event"]).copy()
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype(int)
df["text"] = df["text"].astype(str)

# --- Event-held-out split ---
test_df = df[df["event"] == HOLDOUT_EVENT].copy()
trainval_df = df[df["event"] != HOLDOUT_EVENT].copy()

if len(test_df) == 0:
    raise ValueError(f"No rows found for HOLDOUT_EVENT={HOLDOUT_EVENT}. Check event names:\n{df['event'].value_counts()}")

train_df, val_df = train_test_split(
    trainval_df,
    test_size=0.1,
    random_state=SEED,
    stratify=trainval_df["label"]
)

print("Holdout:", HOLDOUT_EVENT)
print("Train events:", train_df["event"].nunique(), " Val events:", val_df["event"].nunique(), " Test events:", test_df["event"].nunique())
print("Sizes:", len(train_df), len(val_df), len(test_df))
print("Label dist train:\n", train_df["label"].value_counts())
print("Label dist test:\n", test_df["label"].value_counts())

Holdout: Summer2020
Train events: 7  Val events: 7  Test events: 1
Sizes: 531105 59012 76966
Label dist train:
 label
0    339457
1    191648
Name: count, dtype: int64
Label dist test:
 label
0    46966
1    30000
Name: count, dtype: int64


In [8]:
# === A) CLASS-WEIGHTED LOSS + B) THRESHOLD TUNING (event-aware val) ===
# Drop-in blocks for your current DeBERTa event-holdout notebook.

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)

# -----------------------------
# 0) CONFIG (match your dataset)
# -----------------------------
DATA_PATH = "out/combined_with_event.csv"
TEXT_COL  = "text"
LABEL_COL = "label"
EVENT_COL = "event"

TEST_EVENT = "Roe"     # final test (held out)
VAL_EVENT  = "Utah"    # event-aware validation (held out)  <-- change if you want
SEED = 42

MODEL_NAME = "microsoft/deberta-v3-base"
OUT_DIR = "./deberta_weighted_eventaware"

MAX_LEN = 128
USE_FP16 = False

# -----------------------------
# 1) LOAD + CLEAN
# -----------------------------
df = pd.read_csv(DATA_PATH)
df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
df = df.dropna(subset=[TEXT_COL, LABEL_COL, EVENT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL] != ""].copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df[df[LABEL_COL].isin([0, 1])].copy()

assert TEST_EVENT in set(df[EVENT_COL].unique()), "TEST_EVENT not found"
assert VAL_EVENT in set(df[EVENT_COL].unique()), "VAL_EVENT not found"
assert TEST_EVENT != VAL_EVENT, "TEST_EVENT and VAL_EVENT must be different"

print("Rows after cleaning:", len(df))
print("Events:\n", df[EVENT_COL].value_counts())

# -----------------------------
# 2) EVENT-AWARE SPLIT
#    - test: TEST_EVENT only
#    - val:  VAL_EVENT only
#    - train: all remaining events
# -----------------------------
test_df = df[df[EVENT_COL] == TEST_EVENT].copy()
val_df  = df[df[EVENT_COL] == VAL_EVENT].copy()
train_df = df[(df[EVENT_COL] != TEST_EVENT) & (df[EVENT_COL] != VAL_EVENT)].copy()

# Optional: if you want a bit more val data, you can also add a small random slice
# from train into val, but keep it simple for now (pure event-aware val).

print("\nSplit sizes:")
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("\nTrain label dist:\n", train_df[LABEL_COL].value_counts())
print("Val label dist:\n", val_df[LABEL_COL].value_counts())
print("Test label dist:\n", test_df[LABEL_COL].value_counts())

# -----------------------------
# 3) Build HF Datasets (Trainer expects column name 'labels')
# -----------------------------
train_df = train_df[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: "labels"})
val_df   = val_df[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: "labels"})
test_df  = test_df[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: "labels"})

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize(batch):
    return tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)

ds_tok = ds.map(tokenize, batched=True, remove_columns=[TEXT_COL])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# -----------------------------
# 4) A) CLASS-WEIGHTED LOSS (custom Trainer)
#    weights computed from TRAIN ONLY
# -----------------------------

counts = pd.Series(train_df["labels"]).value_counts().sort_index()
counts = counts.reindex([0, 1], fill_value=1)

N = counts.sum()
w0 = N / (2.0 * counts[0])
w1 = N / (2.0 * counts[1])

class_weights = torch.tensor([w0, w1], dtype=torch.float)
print("Class weights:", class_weights.tolist())


class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    # ✅ accept **kwargs to handle HF versions that pass num_items_in_batch
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# -----------------------------
# 5) Training args (keep similar to yours)
# -----------------------------
args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=2,
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=200,
    save_strategy="epoch",
    eval_strategy="epoch" if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames else "epoch",
    fp16=USE_FP16,
    report_to="none",
)

# Keep your compute_metrics if you want; here we keep it minimal and stable
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = (preds == labels).mean()
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

# sanity loss check
batch = next(iter(torch.utils.data.DataLoader(ds_tok["train"], batch_size=8, collate_fn=data_collator)))
batch = {k: v.to(model.device) for k, v in batch.items()}
with torch.no_grad():
    out = model(**batch)
print("\nSanity loss:", float(out.loss))

trainer.train()

# -----------------------------
# 6) B) THRESHOLD TUNING on EVENT-AWARE VALIDATION
#    Use model probabilities and choose threshold that maximizes F1 (or recall)
# -----------------------------
val_out = trainer.predict(ds_tok["validation"])
val_logits = val_out.predictions
val_labels = np.array(ds_tok["validation"]["labels"])

val_probs = torch.softmax(torch.tensor(val_logits), dim=-1).numpy()
val_ai_prob = val_probs[:, 1]

def metrics_at_threshold(y_true, ai_prob, thr):
    y_pred = (ai_prob >= thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    acc = (y_pred == y_true).mean()
    return acc, p, r, f1

thresholds = np.linspace(0.05, 0.95, 19)
rows = []
for t in thresholds:
    acc, p, r, f1 = metrics_at_threshold(val_labels, val_ai_prob, t)
    rows.append((t, acc, p, r, f1))

thr_df = pd.DataFrame(rows, columns=["threshold","acc","precision","recall","f1"]).sort_values("f1", ascending=False)
best_thr = float(thr_df.iloc[0]["threshold"])

print("\nThreshold tuning on event-aware VALIDATION (held-out event):", VAL_EVENT)
print(thr_df.head(10).to_string(index=False))
print("\nBest threshold (max F1 on val):", best_thr)

# -----------------------------
# 7) Evaluate on TEST with:
#    - default threshold 0.5 (argmax equivalent)
#    - tuned threshold best_thr
# -----------------------------
test_out = trainer.predict(ds_tok["test"])
test_logits = test_out.predictions
test_labels = np.array(ds_tok["test"]["labels"])

test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
test_ai_prob = test_probs[:, 1]

# Default 0.5 threshold
pred_default = (test_ai_prob >= 0.5).astype(int)
print("\n=== TEST (held-out event):", TEST_EVENT, " | Default threshold = 0.5 ===")
print("Confusion Matrix:\n", confusion_matrix(test_labels, pred_default))
print(classification_report(test_labels, pred_default, digits=4))

# Tuned threshold from validation event
pred_tuned = (test_ai_prob >= best_thr).astype(int)
print("\n=== TEST (held-out event):", TEST_EVENT, f" | Tuned threshold = {best_thr:.2f} (selected on {VAL_EVENT}) ===")
print("Confusion Matrix:\n", confusion_matrix(test_labels, pred_tuned))
print(classification_report(test_labels, pred_tuned, digits=4))

trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("\n✅ Saved to", OUT_DIR)

Rows after cleaning: 667083
Events:
 event
CapitolAttack    177512
Election2024     170353
Covid            112501
Summer2020        76966
Midterm           47827
Election2020      36457
Roe               35335
Utah              10132
Name: count, dtype: int64

Split sizes:
Train: 621616 Val: 10132 Test: 35335

Train label dist:
 label
0    408554
1    213062
Name: count, dtype: int64
Val label dist:
 label
0    5213
1    4919
Name: count, dtype: int64
Test label dist:
 label
1    24961
0    10374
Name: count, dtype: int64


Map: 100%|██████████| 35335/35335 [00:10<00:00, 3306.15 examples/s]
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class weights: [0.7607513070106506, 1.4587678909301758]


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.



Sanity loss: 0.7097336649894714


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000000,0.359681,0.955685,0.999330,0.909331,0.952209
2,0.000000,1.267972,0.864686,1.000000,0.721285,0.838077



Threshold tuning on event-aware VALIDATION (held-out event): Utah
 threshold      acc  precision   recall       f1
      0.05 0.907126        1.0 0.808701 0.894234
      0.10 0.898539        1.0 0.791014 0.883314
      0.15 0.891532        1.0 0.776581 0.874242
      0.20 0.887683        1.0 0.768652 0.869195
      0.25 0.882748        1.0 0.758487 0.862659
      0.30 0.878405        1.0 0.749543 0.856844
      0.35 0.874161        1.0 0.740801 0.851104
      0.40 0.870904        1.0 0.734092 0.846659
      0.45 0.867943        1.0 0.727993 0.842588
      0.50 0.864686        1.0 0.721285 0.838077

Best threshold (max F1 on val): 0.05



=== TEST (held-out event): Roe  | Default threshold = 0.5 ===
Confusion Matrix:
 [[10366     8]
 [18022  6939]]
              precision    recall  f1-score   support

           0     0.3652    0.9992    0.5349     10374
           1     0.9988    0.2780    0.4349     24961

    accuracy                         0.4897     35335
   macro avg     0.6820    0.6386    0.4849     35335
weighted avg     0.8128    0.4897    0.4643     35335


=== TEST (held-out event): Roe  | Tuned threshold = 0.05 (selected on Utah) ===
Confusion Matrix:
 [[10363    11]
 [17044  7917]]
              precision    recall  f1-score   support

           0     0.3781    0.9989    0.5486     10374
           1     0.9986    0.3172    0.4814     24961

    accuracy                         0.5173     35335
   macro avg     0.6884    0.6581    0.5150     35335
weighted avg     0.8164    0.5173    0.5012     35335


✅ Saved to ./deberta_weighted_eventaware
